In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\HP\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [40]:
def read_amazon_data(amazon_file_path):
    return pd.read_csv(amazon_file_path).dropna(subset=['cleaned_review', 'sentiments'])

In [41]:
def preprocess_amazon_reviews(amazon_reviews):
    amazon_reviews_after_removing_stopwords = []
    stop_words = set(stopwords.words('english'))
    for rev in amazon_reviews:
        tokens = word_tokenize(rev.lower())
        filtered_tokens = [word for word in tokens if word not in stop_words]
        amazon_reviews_after_removing_stopwords.append(' '.join(filtered_tokens))

    return amazon_reviews_after_removing_stopwords

In [42]:
def encode_labels(labels):
    label_encoder = LabelEncoder()
    label_encoder.fit(labels)
    return label_encoder.transform(labels), label_encoder

In [43]:
def split_data(reviews, labels):
    return train_test_split(reviews, labels, test_size=0.2, random_state=42)

In [44]:
def predict_new_review(new_review, vectorizer, model, label_encoder):
    new_review = preprocess_amazon_reviews([new_review])[0]
    review_vector = vectorizer.transform([new_review])
    numerical_prediction = model.predict(review_vector)[0]
    return label_encoder.inverse_transform([numerical_prediction])[0]

In [45]:
amazon_reviews = read_amazon_data('amazon_reviews.csv')
amazon_reviews_after_removing_stopwords = preprocess_amazon_reviews(amazon_reviews['cleaned_review'])
numeric_labels, label_encoder = encode_labels(amazon_reviews['sentiments'])
train_reviews, test_reviews, train_labels, test_labels = split_data(amazon_reviews_after_removing_stopwords, numeric_labels)

vectorizer = TfidfVectorizer()
vectorized_train_reviews = vectorizer.fit_transform(train_reviews)
vectorized_test_reviews = vectorizer.transform(test_reviews)

svm_classifier = SVC(kernel='linear', random_state=42)
svm_classifier.fit(vectorized_train_reviews, train_labels)
svm_predictions = svm_classifier.predict(vectorized_test_reviews)
print("SVM Classification Report:")
print(classification_report(test_labels, svm_predictions))
logistic_regression_classifier = LogisticRegression(random_state=42)
logistic_regression_classifier.fit(vectorized_train_reviews, train_labels)
logistic_regression_predictions = logistic_regression_classifier.predict(vectorized_test_reviews)
print("Logistic Regression Classification Report:")
print(classification_report(test_labels, logistic_regression_predictions))

SVM Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.47      0.57       315
           1       0.77      0.84      0.80      1248
           2       0.90      0.90      0.90      1905

    accuracy                           0.84      3468
   macro avg       0.80      0.74      0.76      3468
weighted avg       0.84      0.84      0.83      3468

Logistic Regression Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.38      0.51       315
           1       0.77      0.83      0.80      1248
           2       0.89      0.91      0.90      1905

    accuracy                           0.83      3468
   macro avg       0.81      0.71      0.74      3468
weighted avg       0.83      0.83      0.83      3468



In [46]:
print("Enter Your Review: (type 'exit' to quit):\n")

while True:
    user_input = input()
    if user_input.lower() == 'exit':
        break
    predicted_sentiment = predict_new_review(user_input, vectorizer, logistic_regression_classifier, label_encoder)
    print(f"Predicted Sentiment: {predicted_sentiment}\n")
    print('*' * 60 + "\n")

Enter Your Review: (type 'exit' to quit):

Predicted Sentiment: positive

************************************************************

Predicted Sentiment: positive

************************************************************

Predicted Sentiment: negative

************************************************************

Predicted Sentiment: neutral

************************************************************

